# CNN kapasite ve düzenlileştirme karşılaştırması
Reyhan Onbaşı · 202612696008

Üç yapılandırma, aynı eğitim/doğrulama ayrımında üç ortak tohumla karşılaştırılır. Bu keşifsel ek deney önceki test sonuçları görüldükten sonra planlanmıştır. Test verisi değerlendirilmez. Önceki ana deney ve beş tohumlu sonuçlar korunur.

Checkpoint ve yapılandırma seçimi, L2 cezasını içermeyen doğrulama çapraz entropisine dayanır. Böylece farklı L2 katsayılarının toplam kayba eklediği ceza karşılaştırmayı doğrudan etkilemez. Yeni bir bağımsız genelleme iddiası kurulmaz.

In [ ]:
import os
os.environ['TF_DETERMINISTIC_OPS'] = '1'
import io, json, zipfile, hashlib, uuid, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, f1_score, log_loss
from google.colab import files
from IPython.display import display
tf.config.experimental.enable_op_determinism()
SEEDS = [101, 202, 303]
CONFIGS = [
 {'name':'Referans', 'width':32, 'dropout':0.30, 'l2':0.0001},
 {'name':'Kucuk_CNN', 'width':16, 'dropout':0.30, 'l2':0.0001},
 {'name':'Guclu_duzenlilestirme', 'width':32, 'dropout':0.40, 'l2':0.0003}]
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:8]
OUT = Path('cnn_dogrulama') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)


## 1. Veri ayrımı ve deney planı

In [ ]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Tek bir kararlılık arşivi gereklidir.')
source_bytes = next(iter(uploaded.values()))
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
    def read_member(name):
        matches=[n for n in z.namelist() if Path(n).name==name]
        if len(matches)!=1:
            raise ValueError(f'{name}: beklenen dosya sayısı 1, bulunan {len(matches)}')
        return z.read(matches[0])
    previous_plan=json.loads(read_member('experiment_plan.json'))
    split_bytes=read_member('split_indices.npz')
with np.load(io.BytesIO(split_bytes), allow_pickle=False) as split:
    train_idx, val_idx, test_idx=[split[k].copy() for k in ['train','validation','test']]
assert [len(train_idx),len(val_idx),len(test_idx)] == [1257,270,270]
indices=np.concatenate([train_idx,val_idx,test_idx])
assert np.issubdtype(indices.dtype,np.integer)
assert np.array_equal(np.sort(indices),np.arange(1797))
assert hashlib.sha256(split_bytes).hexdigest()==previous_plan['split_sha256']
digits=load_digits()
x_train=(digits.images[train_idx].astype('float32')/16)[...,None]
x_val=(digits.images[val_idx].astype('float32')/16)[...,None]
y_train,y_val=digits.target[train_idx],digits.target[val_idx]
(OUT/'split_indices.npz').write_bytes(split_bytes)
plan={'configs':CONFIGS,'seeds':SEEDS,'max_epochs':150,'batch_size':32,
      'learning_rate':0.001,'early_stopping_patience':12,'lr_patience':5,
      'checkpoint_metric':'val_ce','configuration_selection':'lowest mean validation CE; ties: fewer parameters, then name',
      'test_evaluated':False,'planned_after_prior_test_review':True,
      'source_sha256':hashlib.sha256(source_bytes).hexdigest(),
      'split_sha256':hashlib.sha256(split_bytes).hexdigest()}
(OUT/'experiment_plan.json').write_text(json.dumps(plan,indent=2),encoding='utf-8')
versions={p:importlib.metadata.version(p) for p in ['tensorflow','keras','numpy','pandas','scikit-learn','matplotlib']}
versions['python']=platform.python_version()
(OUT/'environment.json').write_text(json.dumps(versions,indent=2))
display(pd.DataFrame(CONFIGS))


## 2. Modeller
Küçük modelde filtre ve gizli nöron sayıları yarıya indirilir. Güçlü düzenlileştirme koşulunda dropout ve L2 birlikte değiştirilir; etkileri ayrı ayrı yorumlanamaz.

In [ ]:
def build_model(cfg):
    w=cfg['width']; reg=regularizers.l2(cfg['l2'])
    model=tf.keras.Sequential([
        layers.Input((8,8,1)),
        layers.Conv2D(w,3,padding='same',activation='relu',kernel_regularizer=reg),
        layers.BatchNormalization(),
        layers.Conv2D(w,3,padding='same',activation='relu',kernel_regularizer=reg),
        layers.MaxPooling2D(2), layers.Dropout(cfg['dropout']),
        layers.Conv2D(2*w,3,padding='same',activation='relu',kernel_regularizer=reg),
        layers.GlobalAveragePooling2D(),
        layers.Dense(2*w,activation='relu',kernel_regularizer=reg),
        layers.Dropout(cfg['dropout']),layers.Dense(10,activation='softmax')])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy',tf.keras.metrics.SparseCategoricalCrossentropy(name='ce')])
    return model


## 3. Dokuz eğitim ve doğrulama değerlendirmesi
Eğitim ve doğrulama farkı, seçilen checkpoint ile her iki kümede de tahmin modunda ölçülür. Böylece dropout ve BatchNormalization aynı modda çalışır. Eğitim günlüğündeki kayıplar ayrıca korunur.

In [ ]:
rows=[]
for cfg in CONFIGS:
    for seed in SEEDS:
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        model=build_model(cfg)
        tag=f"{cfg['name']}_{seed}"
        checkpoint=OUT/f'{tag}.keras'
        cb=[callbacks.ModelCheckpoint(str(checkpoint),monitor='val_ce',mode='min',save_best_only=True),
            callbacks.EarlyStopping(monitor='val_ce',mode='min',patience=12,min_delta=0),
            callbacks.ReduceLROnPlateau(monitor='val_ce',mode='min',patience=5,factor=0.5,min_lr=1e-6)]
        print(tag, flush=True)
        history=model.fit(x_train,y_train,validation_data=(x_val,y_val),epochs=150,
                          batch_size=32,callbacks=cb,shuffle=True,verbose=2)
        h={k:list(map(float,v)) for k,v in history.history.items()}
        (OUT/f'history_{tag}.json').write_text(json.dumps(h))
        model=tf.keras.models.load_model(checkpoint)
        pt=model.predict(x_train,verbose=0); pv=model.predict(x_val,verbose=0)
        train_ce=log_loss(y_train,pt,labels=list(range(10)))
        val_ce=log_loss(y_val,pv,labels=list(range(10)))
        best=int(np.argmin(h['val_ce']))
        row={'configuration':cfg['name'],'seed':seed,'parameters':model.count_params(),
             'epochs':len(h['loss']),'best_epoch':best+1,'train_ce':train_ce,'val_ce':val_ce,
             'ce_gap':val_ce-train_ce,'train_accuracy':accuracy_score(y_train,pt.argmax(1)),
             'val_accuracy':accuracy_score(y_val,pv.argmax(1)),
             'val_macro_f1':f1_score(y_val,pv.argmax(1),average='macro'),
             'val_ce_rise_after_best':h['val_ce'][-1]-h['val_ce'][best]}
        rows.append(row)
        pd.DataFrame(rows).to_csv(OUT/'validation_runs.csv',index=False)
        np.save(OUT/f'validation_probabilities_{tag}.npy',pv)
        pd.DataFrame({'sample_index':val_idx,'true_label':y_val,'prediction':pv.argmax(1)}).to_csv(OUT/f'validation_predictions_{tag}.csv',index=False)
assert len(rows)==9
results=pd.DataFrame(rows)
display(results)


## 4. Karşılaştırma ve kayıt
Düşük doğrulama çapraz entropisi temel ölçüttür. Küçük eğitim–doğrulama farkı tek başına iyi model anlamına gelmez; iki kümede de düşük başarı yetersiz öğrenmeye işaret edebilir. Üç tohum, veri ayrımından kaynaklanan belirsizliği ölçmez. Mevcut doğrulama kümesi tekrar kullanıldığından seçim yanlılığı mümkündür.

In [ ]:
summary=results.groupby('configuration').agg(
    val_ce_mean=('val_ce','mean'),val_ce_std=('val_ce','std'),
    val_accuracy_mean=('val_accuracy','mean'),val_accuracy_std=('val_accuracy','std'),
    val_f1_mean=('val_macro_f1','mean'),ce_gap_mean=('ce_gap','mean'),
    parameters=('parameters','first')).reset_index()
summary=summary.sort_values(['val_ce_mean','parameters','configuration'])
summary.to_csv(OUT/'validation_summary.csv',index=False)
display(summary)
selection={'configuration':str(summary.iloc[0]['configuration']),
           'criterion':'mean validation cross entropy','test_evaluated':False,
           'interpretation':'Exploratory validation comparison; no independent superiority claim.'}
(OUT/'selection.json').write_text(json.dumps(selection,indent=2))
fig,axes=plt.subplots(3,3,figsize=(12,10),constrained_layout=True)
for i,cfg in enumerate(CONFIGS):
    for j,seed in enumerate(SEEDS):
        h=json.loads((OUT/f"history_{cfg['name']}_{seed}.json").read_text())
        ax=axes[i,j]; epochs=np.arange(1,len(h['ce'])+1)
        ax.plot(epochs,h['ce'],label='Eğitim CE (eğitim modu)')
        ax.plot(epochs,h['val_ce'],label='Doğrulama CE')
        ax.axvline(np.argmin(h['val_ce'])+1,color='black',ls='--',lw=0.8)
        ax.set(title=f"{cfg['name']} / {seed}",xlabel='Epoch',ylabel='Çapraz entropi')
        ax.grid(alpha=0.2);ax.legend(fontsize=6)
fig.savefig(OUT/'egitim_egrileri.png',dpi=600)
plt.show()
from shutil import make_archive
archive=make_archive(f'YSA_F26_Dogrulama_Karsilastirma_{RUN_ID}','zip',root_dir=OUT)
print('Doğrulama karşılaştırması tamamlandı. Test değerlendirmesi yapılmadı.')
files.download(archive)
